In [ ]:

# llpsmd_Jun_nuc
import MDAnalysis as mda # 2.3.0, by "mda.version.__version__"
# from MDAnalysis.tests.datafiles import PSF, DCD
from MDAnalysis.analysis import contacts, distances
from MDAnalysis import transformations as trans
from scipy import sparse
import nglview as nv
import os
import fnmatch as fm
import numpy as np
import math
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import parmed
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# define functions
from scipy import optimize
from scipy import sparse

import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import wrap, center_in_box



In [ ]:

# define functions
from scipy import optimize
from scipy import sparse

def timeCorr_bool(mats, N, m):
    # mats is a list of bool matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    #cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k] & mats[k+i]).sum()
            # matCorr.append( (mats[k] * mats[k+i]).sum() )
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//25)-1:
            print(f"{cnt*25} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def timeCorr_boolCsr(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    # cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k].multiply(mats[k+i])).sum()
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//10)-1:
            print(f"{cnt*10} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def exp2(t, a, tau1, c):
    return a * np.exp(-t / tau1) + (1 - a) * np.exp(-t / (c*tau1))

def fit_corr(Ct, t, init):
    # fit normalized correlation ( C(0)=1 )
    n= len(t)
    popt2, pcov2 = optimize.curve_fit(
            exp2, t, Ct, sigma= (t+t[1])**(1/3.0),
            bounds=([0,0,1],[1,10*t[-1],math.inf]), p0=init, maxfev=1000000) # sigma= t**0.5,
    
    Ct_fit= np.zeros(n)
    for i in range(n):
        Ct_fit[i]= exp2(t[i], *popt2)
    return Ct_fit, popt2    

def calc_contact(grp1, grp2, d, dims):
    # calculate contact matrix
    dist= contacts.distance_array(grp1.positions, grp2.positions, box=dims)
    contMat_12= contacts.contact_matrix(dist, radius=d)
    return contMat_12


def timeCorr_boolCsr_fromStart(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m = np.zeros(m)
    
    for i in range(m):
        corr_m[i] = (mats[0].multiply(mats[i])).sum()
    
    corrN_m = corr_m / corr_m[0]
    return corrN_m, corr_m


def visul5(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(nucleic or protein ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=2,
                            color='green')
    
    

    view.add_unitcell(color='white')
    return view


In [ ]:

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list
    


###  9_res

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════

systems    = ['2_close', '2_spread', '3_close', '3_spread']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 9        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000     # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering

offsets    = [0,1000, 2000, 3000, 4000]   # multiple trajectory start offsets

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}

for idx, label in enumerate(systems):

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res/equal/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/9_res/equal/{label}/traj-nw.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    # ── Time axis (in ns) — shared across all offsets ─────────────────────────
    tM = dt * np.arange(m_corr)   # ns

    # ── Multi-offset loop: clustering + contacts + correlation ────────────────
    all_corrN  = []   # normalised correlations, one per offset
    all_corr   = []   # raw correlations, one per offset

    for number in offsets:
        traj_start = -(star + number)
        print(f"\n  --- Offset {number} (traj slice [{traj_start}:]) ---")

        # ── Clustering at the start frame of this window ───────────────────
        ts = u.trajectory[traj_start]
        centroid, to_cluster, clustered_list, res_list = clustering(
            u, ts, eps=eps_clust
        )

        residue_list = ' '
        for item in res_list:
            residue_list += str(item) + ' '

        # ── Atom selections (re-derived per offset) ────────────────────────
        idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::3]
        idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")

        # ── Intra-chain mask ───────────────────────────────────────────────
        resids_1   = idp_1.resids
        resids_2   = idp_2.resids
        chain_1    = resids_1 // chain_size
        chain_2    = resids_2 // chain_size
        intra_mask = (chain_1[:, None] == chain_2[None, :])

        print(f"    idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
              f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")

        # ── Contact matrix accumulation ────────────────────────────────────
        contMt = []
        for _ in u.trajectory[traj_start:]:
            contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
            contMat[intra_mask] = False
            contMt.append(sparse.csr_matrix(contMat))

        print(f"    Contact matrix done. ({len(contMt)} frames)")

        # ── Time correlation ───────────────────────────────────────────────
        corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)

        all_corrN.append(corrN)
        all_corr.append(corr)

    # ── Average across offsets ─────────────────────────────────────────────────
    # Stack to (n_offsets, m_corr) then mean over axis 0
    corrN_avg = np.mean(np.stack(all_corrN, axis=0), axis=0)
    corr_avg  = np.mean(np.stack(all_corr,  axis=0), axis=0)
    corrN_std = np.std( np.stack(all_corrN, axis=0), axis=0)   # optional, useful for plotting

    print(f"\n  Averaged over {len(offsets)} offsets: {offsets}")

    # ── Exponential fitting on the averaged correlation ────────────────────────
    corrFit, para = fit_corr(corrN_avg, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':        tM,
        'corrN':     corrN_avg,       # averaged normalised correlation
        'corrN_std': corrN_std,       # std across offsets (for error bars)
        'corr':      corr_avg,        # averaged raw correlation
        'all_corrN': all_corrN,       # per-offset normalised correlations
        'all_corr':  all_corr,        # per-offset raw correlations
        'offsets':   offsets,
        'corrFit':   corrFit,
        'para':      para,
        't1':        t1,
        't2':        t2,
        't_eff':     t_eff,
    }

# ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '9res_2_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════
 
with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")
 
plt.rcParams.update({"font.size": 18})
 
# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")
 
# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")
 
# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════

systems    = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 9        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000     # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering

offsets    = [0,1000, 2000, 3000, 4000,10000,20000]   # multiple trajectory start offsets

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}

for idx, label in enumerate(systems):

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res//{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/9_res/{label}/traj-nw.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    # ── Time axis (in ns) — shared across all offsets ─────────────────────────
    tM = dt * np.arange(m_corr)   # ns

    # ── Multi-offset loop: clustering + contacts + correlation ────────────────
    all_corrN  = []   # normalised correlations, one per offset
    all_corr   = []   # raw correlations, one per offset

    for number in offsets:
        traj_start = -(star + number)
        print(f"\n  --- Offset {number} (traj slice [{traj_start}:]) ---")

        # ── Clustering at the start frame of this window ───────────────────
        ts = u.trajectory[traj_start]
        centroid, to_cluster, clustered_list, res_list = clustering(
            u, ts, eps=eps_clust
        )

        residue_list = ' '
        for item in res_list:
            residue_list += str(item) + ' '

        
        # ── Atom selections (re-derived per offset) ────────────────────────
        idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::2]
        idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")


        # ── Intra-chain mask ───────────────────────────────────────────────
        resids_1   = idp_1.resids
        resids_2   = idp_2.resids
        chain_1    = resids_1 // chain_size
        chain_2    = resids_2 // chain_size
        intra_mask = (chain_1[:, None] == chain_2[None, :])

        print(f"    idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
              f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")

        # ── Contact matrix accumulation ────────────────────────────────────
        contMt = []
        for _ in u.trajectory[traj_start:]:
            contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
            contMat[intra_mask] = False
            contMt.append(sparse.csr_matrix(contMat))

        print(f"    Contact matrix done. ({len(contMt)} frames)")

        # ── Time correlation ───────────────────────────────────────────────
        corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)

        all_corrN.append(corrN)
        all_corr.append(corr)


    # ── Average across offsets ─────────────────────────────────────────────────
    # Stack to (n_offsets, m_corr) then mean over axis 0
    corrN_avg = np.mean(np.stack(all_corrN, axis=0), axis=0)
    corr_avg  = np.mean(np.stack(all_corr,  axis=0), axis=0)
    corrN_std = np.std( np.stack(all_corrN, axis=0), axis=0)   # optional, useful for plotting

    print(f"\n  Averaged over {len(offsets)} offsets: {offsets}")

    # ── Exponential fitting on the averaged correlation ────────────────────────
    corrFit, para = fit_corr(corrN_avg, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':        tM,
        'corrN':     corrN_avg,       # averaged normalised correlation
        'corrN_std': corrN_std,       # std across offsets (for error bars)
        'corr':      corr_avg,        # averaged raw correlation
        'all_corrN': all_corrN,       # per-offset normalised correlations
        'all_corr':  all_corr,        # per-offset raw correlations
        'offsets':   offsets,
        'corrFit':   corrFit,
        'para':      para,
        't1':        t1,
        't2':        t2,
        't_eff':     t_eff,
    }

# ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '9res_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════
 
with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")
 
plt.rcParams.update({"font.size": 18})
 
# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")
 
# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")
 
# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")

### 3_Res

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════

systems    = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 3        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000     # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering

offsets    = [0,1000, 2000, 3000, 4000,10000,20000]   # multiple trajectory start offsets

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}

for idx, label in enumerate(systems):

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/3_res//{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/3_res/{label}/traj_2-nw.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    # ── Time axis (in ns) — shared across all offsets ─────────────────────────
    tM = dt * np.arange(m_corr)   # ns

    # ── Multi-offset loop: clustering + contacts + correlation ────────────────
    all_corrN  = []   # normalised correlations, one per offset
    all_corr   = []   # raw correlations, one per offset

    for number in offsets:
        traj_start = -(star + number)
        print(f"\n  --- Offset {number} (traj slice [{traj_start}:]) ---")

        # ── Clustering at the start frame of this window ───────────────────
        ts = u.trajectory[traj_start]
        centroid, to_cluster, clustered_list, res_list = clustering(
            u, ts, eps=eps_clust
        )

        residue_list = ' '
        for item in res_list:
            residue_list += str(item) + ' '

        # ── Atom selections (re-derived per offset) ────────────────────────
        idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::]
        idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")

        # ── Intra-chain mask ───────────────────────────────────────────────
        resids_1   = idp_1.resids
        resids_2   = idp_2.resids
        chain_1    = resids_1 // chain_size
        chain_2    = resids_2 // chain_size
        intra_mask = (chain_1[:, None] == chain_2[None, :])

        print(f"    idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
              f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")

        # ── Contact matrix accumulation ────────────────────────────────────
        contMt = []
        for _ in u.trajectory[traj_start:]:
            contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
            contMat[intra_mask] = False
            contMt.append(sparse.csr_matrix(contMat))

        print(f"    Contact matrix done. ({len(contMt)} frames)")

        # ── Time correlation ───────────────────────────────────────────────
        corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)

        all_corrN.append(corrN)
        all_corr.append(corr)

    # ── Average across offsets ─────────────────────────────────────────────────
    # Stack to (n_offsets, m_corr) then mean over axis 0
    corrN_avg = np.mean(np.stack(all_corrN, axis=0), axis=0)
    corr_avg  = np.mean(np.stack(all_corr,  axis=0), axis=0)
    corrN_std = np.std( np.stack(all_corrN, axis=0), axis=0)   # optional, useful for plotting

    print(f"\n  Averaged over {len(offsets)} offsets: {offsets}")

    # ── Exponential fitting on the averaged correlation ────────────────────────
    corrFit, para = fit_corr(corrN_avg, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':        tM,
        'corrN':     corrN_avg,       # averaged normalised correlation
        'corrN_std': corrN_std,       # std across offsets (for error bars)
        'corr':      corr_avg,        # averaged raw correlation
        'all_corrN': all_corrN,       # per-offset normalised correlations
        'all_corr':  all_corr,        # per-offset raw correlations
        'offsets':   offsets,
        'corrFit':   corrFit,
        'para':      para,
        't1':        t1,
        't2':        t2,
        't_eff':     t_eff,
    }

# ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '3res_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════
 
with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")
 
plt.rcParams.update({"font.size": 18})
 
# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")
 
# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")
 
# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════
 
systems    = ['2_close', '2_spread', '3_close', '3_spread']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']
 
# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 9        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000   # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering
 
save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)
 
results = {}

for idx, label in enumerate(systems):
    

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

 
    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res/equal/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/9_res/equal/{label}/traj-nw.dcd')
 

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)
 
    # ── Clustering to find dense cluster ──────────────────────────────────────
    ts = u.trajectory[-star]
    centroid, to_cluster, clustered_list, res_list = clustering(u, ts, eps=eps_clust)

    residue_list = ' '
    for item in res_list:
        residue_list += str(item) + ' '

    # ── FIXED: both idp_1 and idp_2 are dilute-phase CA only ─────────────────
    idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::2]
    idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")

    # ── Intra-chain mask ───────────────────────────────────────────────────────
    resids_1   = idp_1.resids
    resids_2   = idp_2.resids
    chain_1    = resids_1 // chain_size
    chain_2    = resids_2 // chain_size
    intra_mask = (chain_1[:, None] == chain_2[None, :])
 
    print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
          f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")


    # ── Contact matrix accumulation ───────────────────────────────────────────
    contMt = []
    for _ in u.trajectory[-star:]:
        contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
        contMat[intra_mask] = False
        contMt.append(sparse.csr_matrix(contMat))

    print(f"  Contact matrix done. ({len(contMt)} frames)")

    # ── Time axis (in ns) ─────────────────────────────────────────────────────
    tM = dt * np.arange(m_corr)   # ns
 
    # ── Time correlation ───────────────────────────────────────────────────────
    corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)
    #print(f"  Time correlation done.  baseline (frozen fraction) = {baseline/corr[0]:.3f}")
 
    # ── Exponential fitting ────────────────────────────────────────────────────
    # FIXED: initial guess a=0.5, not 0
    corrFit, para = fit_corr(corrN, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")
 
    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':      tM,
        'corrN':   corrN,
        'corr':    corr,
        #'baseline': baseline,
        'corrFit': corrFit,
        'para':    para,
        't1':      t1,
        't2':      t2,
        't_eff':   t_eff,
    }

#### ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '9res_2_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════
 
with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")
 
plt.rcParams.update({"font.size": 18})
 
# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")
 
# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")
 
# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")


### 3_res

In [ ]:


systems    = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']
 
# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 3        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000   # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering
 
save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)
 
results = {}

for idx, label in enumerate(systems):
    

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

 
    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/3_res/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/3_res/{label}/traj_2-nw.dcd')
 

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)
 
    # ── Clustering to find dense cluster ──────────────────────────────────────
    ts = u.trajectory[-star]
    centroid, to_cluster, clustered_list, res_list = clustering(u, ts, eps=eps_clust)

    residue_list = ' '
    for item in res_list:
        residue_list += str(item) + ' '

    # ── FIXED: both idp_1 and idp_2 are dilute-phase CA only ─────────────────
    idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::2]
    idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")

    # ── Intra-chain mask ───────────────────────────────────────────────────────
    resids_1   = idp_1.resids
    resids_2   = idp_2.resids
    chain_1    = resids_1 // chain_size
    chain_2    = resids_2 // chain_size
    intra_mask = (chain_1[:, None] == chain_2[None, :])
 
    print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
          f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")


    # ── Contact matrix accumulation ───────────────────────────────────────────
    contMt = []
    for _ in u.trajectory[-star:]:
        contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
        contMat[intra_mask] = False
        contMt.append(sparse.csr_matrix(contMat))

    print(f"  Contact matrix done. ({len(contMt)} frames)")

    # ── Time axis (in ns) ─────────────────────────────────────────────────────
    tM = dt * np.arange(m_corr)   # ns
 
    # ── Time correlation ───────────────────────────────────────────────────────
    corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)
    
 
    # ── Exponential fitting ────────────────────────────────────────────────────
    # FIXED: initial guess a=0.5, not 0
    corrFit, para = fit_corr(corrN, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")
 
    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':      tM,
        'corrN':   corrN,
        'corr':    corr,
        #'baseline': baseline,
        'corrFit': corrFit,
        'para':    para,
        't1':      t1,
        't2':      t2,
        't_eff':   t_eff,
    }

#### ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '3res_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")



In [ ]:
### Dilute systems



systems    = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']
 
# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 3        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000   # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering
 
save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)
 
results = {}

for idx, label in enumerate(systems):
    

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

 
    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/3_res/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/3_res/{label}/traj_2-nw.dcd')
 

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)
 
    # ── Clustering to find dense cluster ──────────────────────────────────────
    ts = u.trajectory[-star]
    centroid, to_cluster, clustered_list, res_list = clustering(u, ts, eps=eps_clust)

    residue_list = ' '
    for item in res_list:
        residue_list += str(item) + ' '

    # ── FIXED: both idp_1 and idp_2 are dilute-phase CA only ─────────────────
    idp_1 = u.select_atoms(f"name CA and not (resid {residue_list})")[::]
    idp_2 = u.select_atoms(f"name CA and not (resid {residue_list})")

    # ── Intra-chain mask ───────────────────────────────────────────────────────
    resids_1   = idp_1.resids
    resids_2   = idp_2.resids
    chain_1    = resids_1 // chain_size
    chain_2    = resids_2 // chain_size
    intra_mask = (chain_1[:, None] == chain_2[None, :])
 
    print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
          f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")


    # ── Contact matrix accumulation ───────────────────────────────────────────
    contMt = []
    for _ in u.trajectory[-star:]:
        contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
        contMat[intra_mask] = False
        contMt.append(sparse.csr_matrix(contMat))

    print(f"  Contact matrix done. ({len(contMt)} frames)")

    # ── Time axis (in ns) ─────────────────────────────────────────────────────
    tM = dt * np.arange(m_corr)   # ns
 
    # ── Time correlation ───────────────────────────────────────────────────────
    corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)
    
 
    # ── Exponential fitting ────────────────────────────────────────────────────
    # FIXED: initial guess a=0.5, not 0
    corrFit, para = fit_corr(corrN, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")
 
    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':      tM,
        'corrN':   corrN,
        'corr':    corr,
        #'baseline': baseline,
        'corrFit': corrFit,
        'para':    para,
        't1':      t1,
        't2':      t2,
        't_eff':   t_eff,
    }

#### ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '3res_diluter_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")



In [ ]:


# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════
 
with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")
 
plt.rcParams.update({"font.size": 18})
 
# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")
 
# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")
 
# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")


In [ ]:
f'../gxg_comps/9_res/equal/2_close/traj-nw.dcd'

In [ ]:
stru = parmed.load_file(f'../gxg_comps/9_res/equal/2_close//output.top')
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru, f'../gxg_comps/9_res/equal/2_close//traj-nw.dcd')

In [ ]:
u.trajectory[-1000]

In [ ]:
systems    = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']


for sys in systems:
    mkdir '/lustre/hdd/LAS/potoyan-lab/vysakh/gxg/9_res/{sys}'
    cp -r 9_res/{sys}/output* 9_res/{sys}/traj-nw.dcd 9_res/{sys}/traj.dcd 9_res/{sys}/traj.dcd  /lustre/hdd/LAS/potoyan-lab/vysakh/gxg/9_res/{sys}
     